In [16]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import joblib

In [17]:
working_dir = "C:/Users/aless/Desktop/UNIBO_local/PhD_lessons/machine_learning/RF_tp_extreme_SEAS5/data/preprocessed_data" 
mslp = xr.open_dataset(f"{working_dir}/mslp.nc")


In [18]:
mslp.info()

xarray.Dataset {
dimensions:
	ensemble_member = 51 ;
	lat = 41 ;
	lon = 51 ;
	id = 6383 ;

variables:
	float32 msl(ensemble_member, lat, lon, id) ;
		msl:GRIB_paramId = 151 ;
		msl:GRIB_dataType = fc ;
		msl:GRIB_numberOfPoints = 2091 ;
		msl:GRIB_typeOfLevel = surface ;
		msl:GRIB_stepUnits = 1 ;
		msl:GRIB_stepType = instant ;
		msl:GRIB_gridType = regular_ll ;
		msl:GRIB_uvRelativeToGrid = 0 ;
		msl:GRIB_NV = 0 ;
		msl:GRIB_Nx = 51 ;
		msl:GRIB_Ny = 41 ;
		msl:GRIB_cfName = air_pressure_at_mean_sea_level ;
		msl:GRIB_cfVarName = msl ;
		msl:GRIB_gridDefinitionDescription = Latitude/Longitude Grid ;
		msl:GRIB_iDirectionIncrementInDegrees = 1.0 ;
		msl:GRIB_iScansNegatively = 0 ;
		msl:GRIB_jDirectionIncrementInDegrees = 1.0 ;
		msl:GRIB_jPointsAreConsecutive = 0 ;
		msl:GRIB_jScansPositively = 0 ;
		msl:GRIB_latitudeOfFirstGridPointInDegrees = 30.0 ;
		msl:GRIB_latitudeOfLastGridPointInDegrees = -10.0 ;
		msl:GRIB_longitudeOfFirstGridPointInDegrees = -30.0 ;
		msl:GRIB_longitudeOfLa

# Primo tentativo di PCA
preciso che questo workflow me l'ha proposto gemini, ma sembra una prassi necessaria.
In the EOF decomposition, it is rucial to compensate the area reduction of the grid cells towards the poles, by computing the squared root of cosine of the latitude. Moreover, we compute the anomalies along the dimensions of the data, for every gridpoint. 

In [20]:
# Data weangling is a common practice in climate science when performing PCA on gridded data. 
# The idea is to give more weight to grid points that represent larger areas of the Earth, which typically occur at lower latitudes, 
# and less weight to grid points that represent smaller areas, which typically occur at higher latitudes. 
# This is often done by multiplying the data by the square root of the cosine of the latitude (in radians).

# Compute anomalies, so that the first EOF is not dominated by the climatology of the system.
mslp_clim = mslp['msl'].mean(dim=['id', 'ensemble_member']).compute()
mslp_anomalies = mslp['msl'] - mslp_clim

# Extract latitude and convert in radians, then compute the area weights.
lat_rad = np.deg2rad(mslp['lat'])
weights = np.sqrt(np.cos(lat_rad))

# Apply weights to the anomalies.
mslp_weighted = mslp_anomalies * weights

# Flatten the data for PCA: combine the 'lat' and 'lon' dimensions into a single 'features' dimension
mslp_flat = mslp_weighted.stack(
    samples = ('id', 'ensemble_member'),
    features = ('lat', 'lon')
).transpose('samples', 'features')


MemoryError: Unable to allocate 649. MiB for an array with shape (51, 41, 51, 6383) and data type bool

In [ ]:
# PCA via truncated SVD
mslp_pca = PCA(n_components=0.85, random_state=42)  # Keep enough components to explain 85% of the variance
PC_raw = mslp_pca.fit_transform(mslp_flat) 
EOF_raw = mslp_pca.components_ 

# normalize EOFs by the square root of the explained variance
pc_std = np.sqrt(mslp_pca.explained_variance_)
pcs_norm = PC_raw / pc_std # input for RF



In [ ]:
# EOF analysis using PCA
mslp_pca = PCA(n_components=0.85, random_state=42)  # Keep enough components to explain 85% of the variance


